In [27]:
import numpy as np
from utils.utils_puzzle_2023_01 import word_to_num_map, word_to_num_map_with_numbers
from utils.import_puzzle_input import load_input
from utils.import_puzzle_input import split_puzzle_input

# Load the autoreload extension
%load_ext autoreload

# Set autoreload to automatically reload modules before executing code
%autoreload 2

In [28]:
puzzle_input = load_input(2023, 1)
puzzle_input_split = split_puzzle_input(puzzle_input)

loaded puzzle input


In [32]:
subset_puzzle_input_split = puzzle_input_split[0:30]
subset_puzzle_input_split

['eightqrssm9httwogqshfxninepnfrppfzhsc',
 'one111jxlmc7tvklrmhdpsix',
 'bptwone4sixzzppg',
 'ninezfzseveneight5kjrjvtfjqt5nineone',
 '58kk',
 '5b32',
 '1dtwo',
 'six7two7sixtwo78',
 'mvhsixpptztjh13sixthree2',
 'six1bqqvrxndt',
 'fourmk5grmqone944nbvtj',
 'twofiveqxfivezpkvfvxt5eightjhnpl',
 'fpfqp7three7',
 'scmlf76ninegjzjkj97two',
 'fivetkhfnnx22',
 'sevenxvbcbsvxr7eighttwo',
 '1hvhqqmrs1bgttshthg6',
 '4bvnccbdh4onefztdrpq62vvbnvpxxvgrngnfjgfk',
 '653spgrvd',
 'sixctlhkjmmxh2fourfivenine37',
 '229mjp3txmqsxxqdbnnnbrtrcctgzseven',
 'jfourdbpcjc39bhglgnine',
 'bvnltxdmsp7twoxzpdjdvkxeight4twothree',
 'jlvcdrkhzh8seven3',
 '418oneeight',
 '53flcrlvqdeight84frmdcsixchcbc',
 '114sixone1eight2',
 'xrbtzbklqsl11',
 'bhfhszrhzgrhsfd2threeseventwosevenoneseven',
 'four9one']

# General thoughts

Note that numbers can be written as text.

# Solutions

## Solution 1: Crude Version

### Explanation of method

For the crude version, we will perform iterated for-loops. 
We will:
1. Initialise a list to store the numbers of each line.
2. Iterate over the lines in `puzzle_input` using `split("\n")`.  
    2.1. We will initialise an empty string to store the numbers that might be found in each line.  
    2.2. For each line, we will use `word_to_num_map` to convert any text-numbers on the line to symbols.  
    2.3. For each symbol/character in the resulting line add it to the empty string of numbers for this line if `.isdigit()` is `True`.  
    2.4. Add the string to the list of numbers.
3.  We will run through the resulting list of number strings.  
    3.1. Check that the string has atleast length 2  
        3.1.1. If the string is shorter consider the first number to be the last as well.  
        3.1.2. Empty strings will be defined to be 0  
    3.2 Take the first and the last number  
    3.3 Convert the resulting first and last number string to an int.  
4. Sum over the resulting list

### Implementation

#### Execution Step 1&2

In [5]:
def get_num_list_raw(puzzle_input, puzzle_part = 1):
    num_list_raw = []
    for line in puzzle_input.split("\n"):
        numbers_in_line = ""
        line_mod = line
        if puzzle_part == 2:
            for key, value in word_to_num_map.items():
                line_mod = line_mod.replace(key, value) # assume text words do not overlap (eg. no 'twone')
        for ch in line_mod:
            if ch.isdigit():
                numbers_in_line += ch
        num_list_raw.append(numbers_in_line)
    return num_list_raw

#### Execution Step 3

In [6]:
def get_first_last_num(num_list_raw):
    num_list_mod = []
    for nums_in_line in num_list_raw:
        if len(num_list_raw) == 0:
            num_list_mod.append(0)
        elif len(num_list_raw) == 1:
            # >=1 could be collected into a single line with [0] + [-1]
            num_list_mod.append(nums_in_line[0] + nums_in_line[0]) 
        else:
            num_list_mod.append(nums_in_line[0] + nums_in_line[-1])
    first_last_num = [int(num) for num in num_list_mod]
    return first_last_num

#### Execution Step 4

In [7]:
# !!! 25_08_04: ANSWER IS WRONG. 53201 IS TOO LOW!
num_list_raw = get_num_list_raw(puzzle_input, puzzle_part = 1)
first_last_num = get_first_last_num(num_list_raw)
np.sum(first_last_num)

np.int64(54159)

### Checking runtime

In [8]:
%load_ext line_profiler
from profile_dir.aoc_2023_01_profilable import aoc_2023_01_sol_1_func

#### Benchmarking

In [9]:
%timeit aoc_2023_01_sol_1_func(puzzle_input)

2.14 ms ± 302 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


#### Runtime profiling

In [10]:
%lprun -f aoc_2023_01_sol_1_func aoc_2023_01_sol_1_func(puzzle_input)

Timer unit: 1e-09 s

Total time: 0.0324614 s
File: /home/zjh/dev/repos/aoc_2023/profile_dir/aoc_2023_01_profilable.py
Function: aoc_2023_01_sol_1_func at line 5

Line #      Hits         Time  Per Hit   % Time  Line Contents
     5                                           def aoc_2023_01_sol_1_func(puzzle_input):
     6                                               #### Implementation Step 1
     7         1       1272.0   1272.0      0.0      num_list_raw = []
     8                                           
     9                                               #### Implementation Step 2
    10      1001     620584.0    620.0      1.9      for line in puzzle_input.split("\n"):
    11      1000     358200.0    358.2      1.1          numbers_in_line = ""
    12      1000     393811.0    393.8      1.2          line_mod = line
    13     11000    4957189.0    450.7     15.3          for key, value in word_to_num_map.items():
    14                                                       

As expected, we see that most computation time is spent on the for-loop elements.

### Checking solution:

In [11]:
list(puzzle_input.split("\n"))[0:16]

['eightqrssm9httwogqshfxninepnfrppfzhsc',
 'one111jxlmc7tvklrmhdpsix',
 'bptwone4sixzzppg',
 'ninezfzseveneight5kjrjvtfjqt5nineone',
 '58kk',
 '5b32',
 '1dtwo',
 'six7two7sixtwo78',
 'mvhsixpptztjh13sixthree2',
 'six1bqqvrxndt',
 'fourmk5grmqone944nbvtj',
 'twofiveqxfivezpkvfvxt5eightjhnpl',
 'fpfqp7three7',
 'scmlf76ninegjzjkj97two',
 'fivetkhfnnx22',
 'sevenxvbcbsvxr7eighttwo']

In [12]:
num_list_mod[0:10]

NameError: name 'num_list_mod' is not defined

## Solution 2: map

### Implementation

In [ ]:
first_last_num = np.array(list(map(int, num_list_mod)))

## Solution 3: RegEx

## Experimenting

In [ ]:
test_tmp_1 = "onezerotwoone"

In [ ]:
test_tmp_2 = "onezerotwone"

In [ ]:
test_tmp_1.replace("one", "1")

'1zerotwo1'

In [ ]:
test_tmp_2.replace("one", "1")

'1zerotw1'

In [ ]:
test_tmp_2

'onezerotwone'

In [ ]:
help(test_tmp_1.replace)

Help on built-in function replace:

replace(old, new, /, count=-1) method of builtins.str instance
    Return a copy with all occurrences of substring old replaced by new.

      count
        Maximum number of occurrences to replace.
        -1 (the default value) means replace all occurrences.

    If the optional argument count is given, only the first count occurrences are
    replaced.



In [ ]:
test_tmp = "a"

In [ ]:
test_tmp[0] + test_tmp[0]

'aa'

In [ ]:
test_tmp[0] + test_tmp[-1]

'aa'

In [ ]:
test_tmp = [10,20,33,55,75,31, 0]

In [ ]:
np.array(test_tmp)

array([10, 20, 33, 55, 75, 31,  0])

## Part 2

In [ ]:
num_list_raw = get_num_list_raw(puzzle_input, puzzle_part = 2)
first_last_num = get_first_last_num(num_list_raw)
np.sum(first_last_num) # 53201 number too low, probably assumption of no overlapping words is wrong.

np.int64(53201)

For each line, don't replace each written out number, rather find what is the first number, and what is the last number.
- for each line
    - for each written out number
        - get index of written out nummber in line into a list of written out word indices (of length 10, one for each number)
        - Find the location (index of the indices) within the list of written out word indices, with the lowest index
        - If the location within the list with the lowest index were to be `0`, then `0` would be the first number in line, if the location with the lowest index were to be `3` then `3` would be the first number in the line

In [37]:
def part_02(puzzle_input_split: list[str]) -> list:
    numbers = []
    for line in puzzle_input_split:
        first_number = ""
        last_number = ""

        first_occurance = 10**6
        last_occurance = -10
        for key, value in word_to_num_map_with_numbers.items():

            potential_first_occurance = line.find(key)
            potential_last_occurance = line.rfind(key)

            
            if potential_first_occurance != -1 and potential_first_occurance < first_occurance:
                first_occurance = potential_first_occurance
                first_number = value

            if potential_last_occurance != -1 and potential_last_occurance > last_occurance:
                last_occurance = potential_last_occurance
                last_number = value
        combined_first_last_number_of_a_line = first_number + last_number
        if len(combined_first_last_number_of_a_line) == 0:
            numbers.append(0)
        else:
            numbers.append(int(first_number + last_number))
    return numbers

In [ ]:
# first two should be 89 and 16

In [38]:
numbers_subset = part_02(subset_puzzle_input_split)
numbers_subset

[89,
 16,
 26,
 91,
 58,
 52,
 12,
 68,
 62,
 61,
 44,
 28,
 77,
 72,
 52,
 72,
 16,
 42,
 63,
 67,
 27,
 49,
 73,
 83,
 48,
 56,
 12,
 11,
 27,
 41]

In [39]:
np.sum(part_02(puzzle_input_split))

np.int64(53866)